In [ ]:
#Autoencoder for network devices communication based on TLS flows attributes
#second version with new added attributes and values for other ones
#version for cross-validation between several devices
#ALPN
#csv
#sext - '0005'
#ccs - '003C','003D','C023','C024','C027','C028'
#ssv

# Install Python libraries

Install the following Python libraries if not already available in the current kernel.

In [ ]:
!pip install matplotlib
!pip install tensorflow
!pip install scikit-learn

# Dataset Preparation

The data are read from all samples in the given folder (desktop applications). From the source JSON we extract only the following (numerical) features:

* Flow Data: A subset of columns that contain numerical flow metrics ('BytesOut', 'PacketsOut', 'BytesIn', 'PacketsIn', 'Duration') is extracted from the DataFrame. These values are converted to a NumPy array of type float32.
* TLS Records size sequence: The 'RecordSequence' column, which holds sequences (arrays) of integers is taken and the array is padded or truncated so that its length is exactly RECORD_SEQUENCE_SIZE (20). These values are converted to a NumPy array of type float32.

The processed numerical flow data and the padded record sequences are concatenated horizontally (column-wise) such that each row has exactly 25 columns. Any NaN values in the concatenated dataset are replaced with 0.

For each column in the dataset, the minimum and maximum values are computed. A min-max normalization is then applied column-wise. This rescales every value so that each feature (column) lies in the range [0,1].

`normal_df` is prepared and can be used for the autoencoder training.

In [ ]:
#
# This is just ot make the visualization nicer...
#
import math

def is_composite(x):
    """Return True if x is composite (not prime) and x >= 4; otherwise False."""
    if x < 4:
        return False  # 2 and 3 are prime; 1 is neither prime nor composite
    for i in range(2, int(math.sqrt(x)) + 1):
        if x % i == 0:
            return True
    return False

def find_nearest_composites(n):
    """Return the composite numbers greater than n."""
    candidates = []
    for i in range(n, int(n * 3 / 2)):
        if is_composite(i):
            candidates.append(i)
    return candidates

def greatest_divisor_pair(x):
    """
    Return the pair of divisors (d, x//d) for composite x such that
    d is the greatest divisor not exceeding sqrt(x). This pair is closest to each other.
    """
    d = int(math.sqrt(x))
    while d > 1:
        if x % d == 0:
            return (d, x // d)
        d -= 1
    return (1, x)

def get_padding_and_dim(x):
    dif = x
    val_x = x
    val_d1 = 0
    val_d2 = 0
    for nearest in find_nearest_composites(x):
        d1, d2 = greatest_divisor_pair(nearest)  
        if (math.fabs(d1-d2) > dif):
            return (val_x, val_d1, val_d2)
        else:
            val_x = nearest
            val_d1 = d1
            val_d2 = d2
            dif = math.fabs(d1-d2) 

In [ ]:
import json
import glob
import pandas as pd
import numpy as np
from array import array
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler 
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

RECORD_SEQUENCE_SIZE=20
tls_columns_names = np.array([f"tls.rec.{i}" for i in range(RECORD_SEQUENCE_SIZE)])

# Resize row in the array
def resize_row(row, maxlen, pad_value=0):
    current_length = len(row)
    if current_length < maxlen:
        # Calculate the amount of padding needed
        pad_width = maxlen - current_length
        # Pad at the end (you can also pad at the beginning or both sides)
        row = np.pad(row, pad_width=(0, pad_width), mode='constant', constant_values=pad_value)
    else:
        # If the row is longer than the target length, slice it
        row = row[:maxlen]
    return row
# Resize the matrix by padding or removing columns
def pad_sequences(rows, maxlen, pad_value=0):
    resized_rows = [resize_row(row, maxlen) for row in rows]
    return resized_rows

# Loads data from the specified collection of json files. It provides raw data.
def load_json_files(json_files):
    all_data = []
    # Open the file and read each line
    for filename in json_files:
        with open(filename, "r") as file:
            # Use a list comprehension to load each line as a JSON object
            data = [json.loads(line.strip()) for line in file]
            for item in data: all_data.append(item)
    # Convert the list of dictionaries into a DataFrame
    df = pd.DataFrame(all_data)
    df_filtered = df[df['tls.ja3s'] != '']
    return df_filtered

# Extracts features from raw dataset. This will provide suitable output to the preprocessing pipeline.
# Flow related columns: 'BytesOut', 'PacketsOut', 'BytesIn', 'PacketsIn', 'Duration'
# TLS handshake columns: 'TlsClientVersion','TlsServerVersion','TlsServerCipherSuite'
# TLS handshake columns transformed with MultiLabelBinarizer: TlsServerExtensions, TlsClientCipherSuites,
#     TlsClientExtensions, TlsClientSupportedGroups, TlsALPN, TlsClinetSupportedVersions,
#     TlsServerSupportedVerions, 
# TLS record sizes: 'RecordSequence' mapped as 'TlsRecord_X'
#
# The output is a DataFrame with the above specified columns. This dataframe can be used as the input to next
# processing block (preprocessor).
#
def extract_features(df):
    # Flow data
    flow_data = df[['bs', 'ps', 'br', 'pr', 'td']].astype(float)
    # TLS handshake data
    tls_data = df[['tls.cver','tls.sver','tls.scs']].fillna(0).astype(str) 
    # Other TLS attributes - fields of values transformed with MiltiLabelBinarizer
    #selected values are based on the most frequent values in tested datasets
    #tls.sext
    df['tls.sext'] = df['tls.sext'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    sext_possible_values = ['0000','0005','0010','0017','0023','0033','000B','002B','FF01']
    mlb = MultiLabelBinarizer(classes = sext_possible_values)
    mlb.fit([])
    transformed = mlb.transform(df['tls.sext'])
    tls_sext_mlb = pd.DataFrame(transformed,columns=mlb.classes_)
    print(mlb.classes_)
   
    # tls.ccs
    df['tls.ccs'] = df['tls.ccs'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    ccs_possible_values = ['0004','0005','0032','0033','0035','0038','0039','1301','1302','1303','000A',
                           '002F','003C','003D','009C','009D','009E','009F','00FF','C007','C009','C00A','C011','C013',
                           'C014','C023','C024','C027','C028','C02B','C02C','C02F','C030','CC13','CC14','CC15','CCA8','CCA9']
    mlb2 = MultiLabelBinarizer(classes = ccs_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.ccs'])
    tls_ccs_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_ccs_mlb_renamed = tls_ccs_mlb.add_suffix('_ccs')

    # tls.cext
    df['tls.cext'] = df['tls.cext'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    cext_possible_values = ['0000','0005','0010','0012','0015','0017','0023','0033','3374',
                            '4469','000A','000B','000D','001B','002B','002D','FE0D','FF01']
    mlb2 = MultiLabelBinarizer(classes = cext_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.cext'])
    tls_cext_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_cext_mlb_renamed = tls_cext_mlb.add_suffix('_cext')

    #  tls.csg
    df['tls.csg'] = df['tls.csg'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    csg_possible_values = ['0201','0202','0203','0301','0303','0401','0403','0501','0503',
                           '0601','0603','0804','0805','0806','0302','0402','0502','0602']
    mlb2 = MultiLabelBinarizer(classes = csg_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.csg'])
    tls_csg_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_csg_mlb_renamed = tls_csg_mlb.add_suffix('_csg')

    # tls.alpn
    df['tls.alpn'] = df['tls.alpn'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    alpn_possible_values = ['h2','http/1.1','']
    mlb2 = MultiLabelBinarizer(classes = alpn_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.alpn'])
    tls_alpn_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_alpn_mlb_renamed = tls_alpn_mlb.add_suffix('_alpn')
    
    # tls.csv
    df['tls.csv'] = df['tls.csv'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    csv_possible_values = ['0303','0304','']
    mlb2 = MultiLabelBinarizer(classes = csv_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.csv'])
    tls_csv_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_csv_mlb_renamed = tls_csv_mlb.add_suffix('_csv')
    
    # tls.ssv
    df['tls.ssv'] = df['tls.ssv'].apply(
    lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )
    ssv_possible_values = ['0304','']
    mlb2 = MultiLabelBinarizer(classes = ssv_possible_values)
    mlb2.fit([])
    transformed2 = mlb2.transform(df['tls.ssv'])
    tls_ssv_mlb = pd.DataFrame(transformed2,columns=mlb2.classes_)
    tls_ssv_mlb_renamed = tls_ssv_mlb.add_suffix('_ssv')
    
    #zde pridat sloupce i pro dalsi seznamy hodnot
    # TLS records 
    records_data = pd.DataFrame( pad_sequences(df['tls.rec'].values, maxlen=RECORD_SEQUENCE_SIZE), columns=tls_columns_names)
    dataset = pd.concat([flow_data, tls_data, tls_sext_mlb, tls_ccs_mlb_renamed, tls_cext_mlb_renamed, 
                         tls_csg_mlb_renamed, tls_alpn_mlb_renamed, tls_csv_mlb_renamed, tls_ssv_mlb_renamed, records_data], axis=1).fillna(0)
    print(dataset.shape)    
    return dataset
#
# Fits the preprocessor that contains scalers for numerical features and OneHotEncoder for categorical data.
# The result is the Pipeline that can be used for further data processing before they are fed in the Autoencoder.
# 
def fit_preprocessor(df):
    preprocessor = ColumnTransformer(
        transformers=[
            ('num_tls', MinMaxScaler(), tls_columns_names),
            ('num_flow', MinMaxScaler(), ['bs', 'ps', 'br', 'pr', 'td']),           
            ('cat1', OneHotEncoder(categories = [['0303','0301','0300','0302','']], sparse_output=False, handle_unknown='ignore'), ['tls.cver']),
            ('cat2', OneHotEncoder(categories = [['0303','0301','0300','0302','']], sparse_output=False, handle_unknown='ignore'),['tls.sver']),
            ('cat3', OneHotEncoder(categories = [['', '0xc02b', '0xc02f', '0xcc14', '0xcc13', '0xc030', '0xc014',
                                                       '0x0033', '0x0035', '0x009c', '0xc013', '0x0005', '0x009e',
                                                       '0x002f', '0x000a', '0xcca9', '0xcca8', '0x009f', '0xc011',
                                                       '0xc028', '0x009d', '0x0039', '0x1301', '0x1302', '0xc02c',
                                                       '0x003c', '0xc009', '0x0004', '0xc027', '0x003d', '0x0067']], 
                                    sparse_output=False, handle_unknown='ignore'),['tls.scs']),
            ('remaining', 'passthrough', [])         
        ], remainder='passthrough')
    pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
    pipeline.fit(df)
    return pipeline


# -------------------------------------------------
# Load normal data and prepare them for Autoencoder
# Normal data are for now represented as the desktop application communication.
#
raw_df = load_json_files(glob.glob("datasets/homelan.tls/192.168.1.198/*.ndjson"))  #homelan.tls/192.168.1.169/
print(f'dataset shape={raw_df.shape}')

raw_df = raw_df.rename(columns={'te': 'td'}) #needed for homelan datasets - different column name
input_df = extract_features(raw_df)

pipeline = fit_preprocessor(input_df)
normal_df = pipeline.transform(input_df)

print('Normalized row of data:')
print(normal_df[0])

normal_pandas_df = pd.DataFrame(normal_df)
print(normal_pandas_df.head())

print(f'dataset shape={normal_df.shape}')

row_len = normal_df.shape[1]
(newrow_len,IMAGE_DIM_X,IMAGE_DIM_Y) = get_padding_and_dim(row_len)
IMAGE_PAD= newrow_len - row_len
print(f"visualization adjustement: {row_len} -> {newrow_len} (+ {IMAGE_PAD}) [{IMAGE_DIM_X} x {IMAGE_DIM_Y}]")

def make_image_from_sample(sample):
    return np.pad(sample, pad_width=(0,IMAGE_PAD), mode='constant', constant_values=0).reshape(IMAGE_DIM_X, IMAGE_DIM_Y)



# Autoencoder Training

The autoencoder is trained on `normal_df`. The data are split into training (80%) and testing (20%) parts. The size of laten space is set to `LATENT_SPACE_SIZE`.

The results are visualzied for the 10 selected sample. 

Read more: https://www.tensorflow.org/tutorials/generative/autoencoder

k-fold cross-validation used for training and subsequent testing

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv1D, MaxPooling1D, Flatten, Lambda, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns
LATENT_SPACE_SIZE=10

#sequential interleaved k-fold split
def interleaved_kfold_df(df, k):
    """
    Returns a list of (train_idx, val_idx) tuples using interleaved k-fold.
    These are index arrays you can use to extract rows from the original df.
    """
    n_samples = len(df)
    all_indices = np.arange(n_samples)
    fold_indices = [all_indices[i::k] for i in range(k)]

    folds = []
    for i in range(k):
        val_idx = fold_indices[i]
        print(f'Validation set indices: {val_idx}')
        train_idx = np.hstack([fold_indices[j] for j in range(k) if j != i])
        folds.append((train_idx, val_idx))
    
    return folds


def autoencoder_training(x_train, x_test) -> Model:
    # 1. Dataset split done in advance to ensure k-fold cross validation
    print(f'train shape={x_train.shape}')
    print(f'test shape={x_test.shape}')
    
    # 2. Define dimensions and architecture
    input_dim = x_train.shape[1]
    encoding_dim = LATENT_SPACE_SIZE  # Size of the latent space
        
    conv_input_size = 20
    intermediate_dim = 64
    
    # Input placeholder
    input_img = Input(shape=(input_dim,))
    # Slice the first 20 values for convolution
    def slice_first_20(x):
        return tf.reshape(x[:, :conv_input_size], (-1, conv_input_size, 1))  # shape: (batch, steps, channels)
    sliced_input = Lambda(slice_first_20, output_shape=(conv_input_size, 1))(input_img)
    
    # Convolutional processing
    conv_layer = Conv1D(filters=32, kernel_size=3, activation='relu')(sliced_input)
    pool_layer = MaxPooling1D(pool_size=2)(conv_layer)
    flattened_conv = Flatten()(pool_layer)
    
    # Remaining input: slice from index 20 onward
    def slice_remaining(x):
        return x[:, conv_input_size:]
    
    remaining_input = Lambda(slice_remaining, output_shape=(input_dim - conv_input_size,))(input_img)
    dense_input = Dense(intermediate_dim, activation='relu')(remaining_input)
    
    # Combine processed convolutional and dense features
    combined = Concatenate()([flattened_conv, dense_input])
    # Encoder layers
    hidden = Dense(intermediate_dim, activation='relu')(combined)
    encoded = Dense(encoding_dim, activation='relu')(hidden)
    
    # Decoder layers
    hidden_decoded = Dense(intermediate_dim, activation='relu')(encoded)
    decoded = Dense(input_dim, activation='sigmoid')(hidden_decoded)
    
    # Autoencoder model
    autoencoder = Model(input_img, decoded)
    
    # Encoder model for later use
    encoder = Model(input_img, encoded)
    
    # Decoder model setup
    encoded_input = Input(shape=(encoding_dim,))
    decoder_layer1 = autoencoder.layers[-2](encoded_input)
    decoder_layer2 = autoencoder.layers[-1](decoder_layer1)  #zde bylo -1
    decoder = Model(encoded_input, decoder_layer2)
    
    # 3. Compile and train the autoencoder
    autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
    autoencoder.fit(x_train, x_train, validation_data=(x_test, x_test), epochs=100)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    autoencoder.fit(x_train, x_train,
                    epochs=500,                      # use some more reasonable number here (>50)
                    callbacks = [early_stop],
                    batch_size=16,
                    shuffle=True,                
                    validation_data=(x_test, x_test))
    
    # 4. Visualize the reconstructed images
    encoded_imgs = encoder.predict(x_test)
    decoded_imgs = decoder.predict(encoded_imgs)
    
    # Assuming x_test contains the original test data
    # and decoded_imgs are the autoencoder's reconstructed outputs
    reconstruction_errors = np.mean(np.square(x_test - decoded_imgs), axis=1)
    
    # Average reconstruction error across all samples
    avg_error = np.mean(reconstruction_errors)
    max_error = np.max(reconstruction_errors)
    min_error = np.min(reconstruction_errors)
    std_error = np.std(reconstruction_errors)
    print(f"Reconstruction error for each sample {reconstruction_errors}")
    print(f"Average={avg_error}, Max={max_error}, Min={min_error} reconstruction errors.")
    print(f"Standard deviation of reconstruction error = {std_error}.")
    
    #false positives evaluation
    FP_1_sigma = np.sum(reconstruction_errors > (avg_error + 1*std_error))
    print(f"Number of false positives for 1 sigma is {FP_1_sigma} with threshold {avg_error + 1*std_error}.")
    FP_2_sigma = np.sum(reconstruction_errors > (avg_error + 2*std_error))
    print(f"Number of false positives for 2 sigma is {FP_2_sigma}.")
    FP_3_sigma = np.sum(reconstruction_errors > (avg_error + 3*std_error))
    print(f"Number of false positives for 3 sigma is {FP_3_sigma}.")
    
    
    sns.violinplot(data= reconstruction_errors)
    
    # Get indices that would sort the array in ascending order
    worst20 = np.argsort(reconstruction_errors)[-20:][::-1]
    worst50 = np.argsort(reconstruction_errors)[-50:][::-1]
    
    n = 20  # Number of digits to display
    i = 0
    
    print("Worst reconstructed:")
    plt.figure(figsize=(20, 4))
    for j in worst20:
        reconstruction_error = reconstruction_errors[j]
        original = x_test[j]
        original = make_image_from_sample(original) 
        reconstructed = decoded_imgs[j]
        reconstructed = make_image_from_sample(reconstructed) 
        # Original images
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(original, cmap='gray')
        plt.title("Original")
        plt.axis('off')
        
        # Reconstructed images
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(reconstructed, cmap='gray')
        plt.title(f"RE {reconstruction_error:.3f}")
        plt.axis('off')
        i+=1
    plt.show()

    return autoencoder, avg_error, std_error

def test_dataset(model: Model, avg_error, std_error, json_files):
    raw_test_df = load_json_files(json_files)
    raw_test_df = raw_test_df.rename(columns={'te': 'td'})
    print(f'raw dataset shape={raw_test_df.shape}')
    input_test_df = extract_features(raw_test_df)
    test_df = pipeline.transform(input_test_df)

    decoded_test = model.predict(test_df)

    # Assuming x_test contains the original test data
    # and decoded_imgs are the autoencoder's reconstructed outputs
    test_reconstruction_errors = np.mean(np.square(test_df - decoded_test), axis=1)
    print("Test reconstruction shape:")
    print(test_reconstruction_errors.shape)
    
    #Find numbers of FN a TP for different tresholds
    FN_1_sigma = np.sum(test_reconstruction_errors < (avg_error + 1*std_error))
    print(f"Number of false negatives for 1 sigma is {FN_1_sigma} with threshold {avg_error + 1*std_error}.")
    TP_1_sigma = np.sum(test_reconstruction_errors > (avg_error + 1*std_error))
    print(f"Number of true positives for 1 sigma is {TP_1_sigma} with threshold {avg_error + 1*std_error}.")    
    FN_2_sigma = np.sum(test_reconstruction_errors < (avg_error + 2*std_error))
    print(f"Number of false negatives for 2 sigma is {FN_2_sigma} with threshold {avg_error + 2*std_error}.")
    TP_2_sigma = np.sum(test_reconstruction_errors > (avg_error + 2*std_error))
    print(f"Number of true positives for 2 sigma is {TP_2_sigma} with threshold {avg_error + 2*std_error}.")   
    FN_3_sigma = np.sum(test_reconstruction_errors < (avg_error + 3*std_error))
    print(f"Number of false negatives for 3 sigma is {FN_3_sigma} with threshold {avg_error + 3*std_error}.")
    TP_3_sigma = np.sum(test_reconstruction_errors > (avg_error + 3*std_error))
    print(f"Number of true positives for 3 sigma is {TP_3_sigma} with threshold {avg_error + 3*std_error}.")   
       
    # Get indices that would sort the array in ascending order
    worst10 = np.argsort(test_reconstruction_errors)[-10:][::-1]
    worst50 = np.argsort(test_reconstruction_errors)[-50:][::-1]

    # Average reconstruction error across all samples
    test_avg_error = np.mean(test_reconstruction_errors)
    test_min_error = np.min(test_reconstruction_errors)
    test_max_error = np.max(test_reconstruction_errors)
    print("Reconstruction error for each sample:", test_reconstruction_errors)
    print("Average reconstruction error:", test_avg_error)
    print("Min reconstruction error:", test_min_error)
    print("Max reconstruction error:", test_max_error)
    print("Worst reconstructed:")
    n = 10  # Number of digits to display
    i = 0
    plt.figure(figsize=(20, 4))
    for j in worst10:
        rec_error = test_reconstruction_errors[j]       
        original = test_df[j]
        original = make_image_from_sample(original)  
        reconstructed = decoded_test[j]
        reconstructed = make_image_from_sample(reconstructed)
        # Original images
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(original, cmap='gray')
        plt.title("Original")
        plt.axis('off')
        
        # Reconstructed images
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(reconstructed, cmap='gray')
        plt.title(f"RE {rec_error:.3f}")
        plt.axis('off')
        i+=1
    plt.show()
    print(raw_test_df.shape)



k = 10
folds = interleaved_kfold_df(normal_df, k)
_, val_idx = folds[-1]
last_val_df = raw_df.iloc[val_idx]

# Save it to CSV
#last_val_df.to_csv('last_validation_set.csv', index=False)
#last_val_df.to_csv('val9.csv')

for fold_idx, (train_idx, test_idx) in enumerate(folds):
    print(f"\n🌀 Fold {fold_idx + 1}/{k}")
    x_train = normal_df[train_idx]
    x_test = normal_df[test_idx]
    autoencoder, avg_error, std_error = autoencoder_training(x_train,x_test)
    print("cross-val dataset: 192.168.1.169")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.169/*.ndjson"))
    print("cross-val dataset: 192.168.1.172")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.172/*.ndjson"))   
    print("cross-val dataset: 192.168.1.174")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.174/*.ndjson"))    
    print("cross-val dataset: 192.168.1.181")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.181/*.ndjson"))    
    print("cross-val dataset: 192.168.1.185")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.185/*.ndjson"))
    print("cross-val dataset: 192.168.1.197")
    test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.197/*.ndjson"))
    #print("cross-val dataset: 192.168.1.198")
    #test_dataset(autoencoder, avg_error, std_error, json_files=glob.glob("datasets/homelan.tls/192.168.1.198/*.ndjson"))